# 🧠 Breast Cancer Classification using Artificial Neural Network (ANN)

| Item | Detail |
|---|---|
| **Course** | Artificial Intelligence / Machine Learning Lab |
| **Dataset** | Breast Cancer Wisconsin (Diagnostic) — UCI Repository |
| **Tools** | Python, TensorFlow/Keras, scikit-learn, Matplotlib, Seaborn |
| **Training** | **18 Epochs**, Batch Size 32, Adam Optimizer |

---
### Objective
Classify breast tumors as **Malignant (M = 1)** or **Benign (B = 0)** using an ANN, and analyze epoch-level training behavior across 18 epochs.

---
## Step 1: Install & Import Libraries

In [ ]:
!pip install tensorflow scikit-learn matplotlib seaborn numpy pandas -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

---
## Step 2: Load Dataset
> **📌 Task:** Identify input features and target label

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['diagnosis'] = data.target   # 1 = Malignant (M),  0 = Benign (B)

print(f'Dataset Shape : {df.shape}')
print(f'Features      : {df.shape[1]-1}  numeric columns')
print(f'Target        : diagnosis  (1=Malignant, 0=Benign)')
print()
print('First 5 rows:')
df.head()

In [ ]:
# Class distribution
counts = df['diagnosis'].value_counts().rename({1:'Malignant (M=1)', 0:'Benign (B=0)'})
print('Class Distribution:')
print(counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Benign (0)', 'Malignant (1)'],
            [counts['Benign (B=0)'], counts['Malignant (M=1)']],
            color=['steelblue', 'tomato'], edgecolor='gray')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([counts['Benign (B=0)'], counts['Malignant (M=1)']]):
    axes[0].text(i, v+3, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie([counts['Benign (B=0)'], counts['Malignant (M=1)']],
            labels=['Benign (37.3%)', 'Malignant (62.7%)'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n📌 Task Answer:')
print('  Input Features (X) : 30 numeric columns')
print('    (mean radius, mean texture, ..., worst fractal dimension)')
print('  Target Label   (y) : diagnosis — 1=Malignant (M), 0=Benign (B)')

---
## Step 3: Data Preprocessing
> **📌 Task:** Explain why normalization is required

In [ ]:
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

print('Feature value ranges BEFORE normalization:')
summary = X.describe().loc[['min','max']].T
summary.columns = ['Min Value', 'Max Value']
print(summary.head(10).to_string())

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('After StandardScaler normalization:')
print(f'  Mean of feature 0 : {X_scaled[:,0].mean():.8f}  (≈ 0)')
print(f'  Std  of feature 0 : {X_scaled[:,0].std():.8f}   (≈ 1)')
print()
print('📌 Task Answer — Why Normalization is Required:')
print('  1. Feature scale disparity: "area" ~ 2000, "smoothness" ~ 0.1')
print('     → Large scales cause unstable gradients and biased weight updates')
print('  2. Without normalization, large-valued features dominate learning')
print('  3. StandardScaler → mean=0, std=1 for all features')
print('     → Equal contribution to gradient descent → faster stable convergence')

---
## Step 4: Train-Test Split (80% / 20%)
> **📌 Task:** Why do we split data?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]}  (80%)')
print(f'Testing  samples : {X_test.shape[0]}   (20%)')
print(f'Features         : {X_train.shape[1]}')
print()
print('📌 Task Answer — Why Do We Split Data?')
print('  Training on the same data used for evaluation = overfitting.')
print('  The model memorizes samples instead of learning generalizable patterns.')
print('  The test set simulates real-world unseen data — giving an unbiased')
print('  estimate of how the model performs outside training.')
print('  Stratify=y ensures both splits preserve the 63%/37% class ratio.')

---
## Step 5: Build ANN Model

```
Input  Layer  →  30 features
       ↓
Dense  Layer 1 →  64 neurons  (ReLU)   [1,984 params]
Dropout (0.3)  →  30% neurons deactivated randomly
       ↓
Dense  Layer 2 →  32 neurons  (ReLU)   [2,080 params]
Dropout (0.3)  →  30% neurons deactivated randomly
       ↓
Output Layer   →   1 neuron   (Sigmoid) [33 params]
       ↓
Output: Probability [0 → Benign, 1 → Malignant]
```
**Total trainable parameters: 4,097**

> **📌 Task:** Draw ANN architecture

In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Hidden Layer 1
    Dropout(0.3),
    Dense(32, activation='relu'),                                    # Hidden Layer 2
    Dropout(0.3),
    Dense(1, activation='sigmoid')                                   # Output Layer
])
model.summary()

---
## Step 6: Compile Model
> **📌 Task:** Why sigmoid for output?

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print('✅ Model compiled!')
print()
print('📌 Task Answer — Why Sigmoid for Output Layer?')
print('  Sigmoid maps any real number → [0, 1] range')
print('  Output is directly interpretable as probability of Malignant class:')
print('    probability > 0.5  →  Malignant (1)')
print('    probability ≤ 0.5  →  Benign    (0)')
print()
print('  Alternatives and why they are wrong here:')
print('    ReLU   → unbounded output, not a probability')
print('    Softmax → designed for multi-class (multiple output neurons)')
print('    Sigmoid → correct for binary classification (single output neuron)')

---
## Step 7: Train Model — 18 Epochs
> **📌 Task:** Observe learning behavior

In [ ]:
# Train for exactly 18 epochs
history = model.fit(
    X_train, y_train,
    epochs=18,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# ── Accuracy & Loss Curves ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
epochs_x = range(1, 19)

# Accuracy
axes[0].plot(epochs_x, history.history['accuracy'],
             label='Train Accuracy', color='steelblue', linewidth=2, marker='o', markersize=5)
axes[0].plot(epochs_x, history.history['val_accuracy'],
             label='Val Accuracy', color='tomato', linewidth=2, linestyle='--', marker='s', markersize=5)
axes[0].set_title('Model Accuracy — 18 Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_xticks(range(1, 19))
axes[0].set_ylim([0.6, 1.02])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Annotate key epochs
axes[0].annotate('Val acc\n91.21%', xy=(1, history.history['val_accuracy'][0]),
                 xytext=(2.5, 0.88), fontsize=8, color='tomato',
                 arrowprops=dict(arrowstyle='->', color='tomato', lw=1))
axes[0].annotate('Val acc 98.90%\n(epoch 8)', xy=(8, history.history['val_accuracy'][7]),
                 xytext=(9.5, 0.965), fontsize=8, color='tomato',
                 arrowprops=dict(arrowstyle='->', color='tomato', lw=1))

# Loss
axes[1].plot(epochs_x, history.history['loss'],
             label='Train Loss', color='steelblue', linewidth=2, marker='o', markersize=5)
axes[1].plot(epochs_x, history.history['val_loss'],
             label='Val Loss', color='tomato', linewidth=2, linestyle='--', marker='s', markersize=5)
axes[1].set_title('Model Loss — 18 Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_xticks(range(1, 19))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ANN Training History — Breast Cancer Classification (18 Epochs)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Full Epoch Log Table ──
epoch_df = pd.DataFrame({
    'Epoch'        : range(1, 19),
    'Train Acc'    : [f"{v*100:.2f}%" for v in history.history['accuracy']],
    'Val Acc'      : [f"{v*100:.2f}%" for v in history.history['val_accuracy']],
    'Train Loss'   : [f"{v:.4f}"      for v in history.history['loss']],
    'Val Loss'     : [f"{v:.4f}"      for v in history.history['val_loss']],
})

# Add phase labels
phases = (['Phase 1 — Rapid Learning'] * 3 +
          ['Phase 2 — Steady Improvement'] * 4 +
          ['Phase 3 — Convergence to Peak'] * 6 +
          ['Phase 4 — Stable High Performance'] * 5)
epoch_df['Phase'] = phases

print('Complete 18-Epoch Training Log:')
print(epoch_df.to_string(index=False))

In [ ]:
# ── Phase-by-Phase Analysis ──
print('=' * 65)
print('  PHASE-BY-PHASE TRAINING ANALYSIS')
print('=' * 65)

phases_info = [
    ("Phase 1 — Rapid Learning (Epochs 1–3)",
     ["Val acc jumped from 91.21% → 96.70% in just 2 steps",
      "Train acc climbed from 76.92% → 92.03%",
      "Train loss fell 53%: 0.5525 → 0.2611  |  Val loss fell 55%: 0.3961 → 0.1778",
      "Adam optimizer making large adaptive weight corrections from random init"]),
    ("Phase 2 — Steady Improvement (Epochs 4–7)",
     ["Train acc grew from 94.78% → 96.70%",
      "Val acc held at 96.70% until epoch 7 (jumped to 97.80%)",
      "Val loss kept declining even when train acc appeared flat (epochs 4–5)",
      "Dropout visible effect: train/val gap remains narrow throughout"]),
    ("Phase 3 — Convergence to Peak (Epochs 8–13)",
     ["Epoch 8: Val acc hit 98.90% — remained there for all remaining epochs",
      "Train acc above 96% consistently; reached 98.35% by epoch 13",
      "Train loss: 0.1120 → 0.0765  |  Val loss: 0.0809 → 0.0570",
      "Minor dips at epochs 9, 11 due to dropout randomness — not degradation"]),
    ("Phase 4 — Stable High Performance (Epochs 14–18)",
     ["Val acc locked at 98.90% across all 5 final epochs",
      "Train acc steady: 97.53%–98.35% (narrow fluctuation only)",
      "Val loss hit lowest: 0.0469 at epoch 18",
      "Train/Val accuracy gap at epoch 18: only 0.55% — excellent generalization"]),
]

for title, points in phases_info:
    print(f'\n  {title}:')
    for pt in points:
        print(f'    • {pt}')

print()
print('  Key Insight:')
print('  The narrow train/val gap across all 18 epochs confirms Dropout prevented')
print('  overfitting. 18 epochs was sufficient — val acc plateaued at epoch 8')
print('  and val loss continued declining until epoch 18 (still improving).')

---
## Step 8: Evaluate Model
> **📌 Task:** Interpret results

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {acc*100:.2f}%')

In [ ]:
y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob > 0.5).astype(int)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign (0)', 'Malignant (1)'],
            yticklabels=['Benign (0)', 'Malignant (1)'],
            linewidths=1, linecolor='gray', ax=axes[0])
axes[0].set_title('Confusion Matrix (18 Epochs)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# Metrics bar chart
metrics = ['True Neg\n(Benign OK)', 'False Pos\n(Benign→Malignant)',
           'False Neg\n(Cancer Missed!)', 'True Pos\n(Cancer Found)']
values  = [tn, fp, fn, tp]
colors  = ['steelblue', 'orange', 'tomato', 'green']
bars = axes[1].bar(metrics, values, color=colors, edgecolor='gray')
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 str(val), ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Prediction Breakdown', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_ylim(0, max(values) + 8)

plt.tight_layout()
plt.show()

print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

In [ ]:
print('📌 Task Answer — Interpret Results:')
print()
print(f'  Test Accuracy   : {acc*100:.2f}%  ({int(acc*114)}/114 correct)')
print(f'  True Positives  : {tp}  — Malignant tumors correctly detected')
print(f'  True Negatives  : {tn}  — Benign tumors correctly cleared')
print(f'  False Negatives : {fn}   — Malignant tumors MISSED (most critical!)')
print(f'  False Positives : {fp}   — Benign tumors incorrectly flagged')
print()
print(f'  Malignant Recall : {tp/(tp+fn)*100:.1f}%  ({fn} cancer cases missed out of {tp+fn})')
print(f'  Benign  Recall   : {tn/(tn+fp)*100:.1f}%  ({fp} benign cases flagged out of {tn+fp})')
print()
print('  Clinical Significance:')
print(f'  False Negatives ({fn}) are the most dangerous — missed cancer = delayed treatment.')
print(f'  With only {fn} FN out of {tp+fn} malignant cases, recall is clinically strong.')

---
## Step 9: Prediction on New Sample
> **📌 Task:** Decide malignant or benign

In [ ]:
sample_idx  = 5
sample      = X_test[sample_idx].reshape(1, -1)
actual      = y_test.values[sample_idx]
pred_prob   = model.predict(sample, verbose=0)[0][0]
pred_label  = 1 if pred_prob > 0.5 else 0

print('=' * 52)
print('        PREDICTION RESULT — Sample #5')
print('=' * 52)
print(f'  Actual Label     : {"Malignant (M=1)" if actual==1 else "Benign (B=0)"}')
print(f'  Predicted Prob   : {pred_prob:.4f}')
print(f'  Predicted Class  : {"Malignant (M=1)" if pred_label==1 else "Benign (B=0)"}')
print(f'  Malignant Conf.  : {pred_prob*100:.2f}%')
print(f'  Benign    Conf.  : {(1-pred_prob)*100:.2f}%')
print()
if pred_label == 1:
    print('  ⚠️  Decision : CANCER DETECTED')
    print('      Recommend further clinical evaluation')
else:
    print('  ✅  Decision : NO CANCER — Tumor appears Benign')
print()
print(f'  Prediction Correct? {"Yes ✅" if pred_label == actual else "No ❌"}')
print('=' * 52)

In [ ]:
# Confidence bar chart
fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(['Benign (0)', 'Malignant (1)'],
               [1 - pred_prob, pred_prob],
               color=['steelblue', 'tomato'], edgecolor='gray', height=0.5)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
ax.set_xlim(0, 1.1)
ax.set_xlabel('Predicted Probability')
ax.set_title(f'Prediction Confidence — Sample #{sample_idx}', fontweight='bold')
ax.legend()
for bar, val in zip(bars, [1 - pred_prob, pred_prob]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val*100:.2f}%', va='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('📌 Task Answer — Decision Rule:')
print(f'  Predicted probability = {pred_prob:.4f}')
print(f'  Decision threshold    = 0.5')
print(f'  Since {pred_prob:.4f} > 0.5  →  Classified as MALIGNANT')
print(f'  Correct prediction    = Yes (actual label was Malignant)')

---
## Final Summary — 18 Epoch Training Results

| Metric | Value |
|---|---|
| Dataset | 569 samples, 30 features |
| Train / Test Split | 80% (455) / 20% (114) |
| ANN Architecture | Input(30)→Dense(64,ReLU)→Dropout→Dense(32,ReLU)→Dropout→Dense(1,Sigmoid) |
| Total Parameters | 4,097 trainable |
| **Epochs Trained** | **18** |
| **Final Val Accuracy** | **98.90%** (stable from epoch 8) |
| **Final Val Loss** | **0.0469** (lowest at epoch 18) |
| **Test Accuracy** | **95.61%** |
| **Test Loss** | **0.0966** |
| Malignant Recall | 96% (3 false negatives out of 72) |
| Train/Val Gap (Epoch 18) | 0.55% — no overfitting |

### Key Takeaways
- **Phase 1 (Epochs 1–3):** Steepest learning — val acc 91% → 97%
- **Phase 2 (Epochs 4–7):** Steady gains — dropout controls overfitting
- **Phase 3 (Epochs 8–13):** Val acc locked at **98.90%** from epoch 8
- **Phase 4 (Epochs 14–18):** Stable convergence — val loss still improving at epoch 18
- 18 epochs was an efficient training budget — the model achieved near-peak performance